In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

aayushjamwal03_prompts_path = kagglehub.dataset_download('aayushjamwal03/prompts')
aayushjamwal03_qwen_results_path = kagglehub.dataset_download('aayushjamwal03/qwen-results')
aayushjamwal03_results_eval_suite_path = kagglehub.dataset_download('aayushjamwal03/results-eval-suite')
aayushjamwal03_eval_updated_path = kagglehub.dataset_download('aayushjamwal03/eval-updated')

print('Data source import complete.')


# PROMPTS AND LOADING REQUIRED STUFF

In [ ]:
import os
os.environ["PIP_LOG"] = "/tmp/pip.log"
os.environ["PIP_CACHE_DIR"] = "/tmp/pip_cache"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["TMPDIR"] = "/tmp"

# Install precompiled CUDA 12 wheel for llama-cpp-python (zero torch conflicts)
!pip install -q --log /tmp/pip.log llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

from huggingface_hub import HfApi, hf_hub_download

# Locate and download the Q4_K_M GGUF file (~17 GB) into /tmp
api = HfApi()
gguf_repos = [
    "sarvamai/sarvam-30b-gguf"
]

model_path = None
for repo in gguf_repos:
    try:
        files = api.list_repo_files(repo)
        q4_files = [f for f in files if "Q4_K_M" in f and f.endswith(".gguf")]
        if q4_files:
            print(f"Downloading {q4_files[0]} from {repo}...", flush=True)
            model_path = hf_hub_download(repo_id=repo, filename=q4_files[0], local_dir="/tmp/sarvam_gguf")
            break
    except Exception:
        continue

print("Downloaded GGUF model to:", model_path)

Downloaded GGUF model to: /tmp/sarvam_gguf/sarvam-30b-Q4_K_M.gguf-00001-of-00006.gguf


In [ ]:
import json

with open("/kaggle/input/datasets/aayushjamwal03/prompts/prompts_english.json", "r", encoding="utf-8") as f:
    english_prompts = json.load(f)

with open("/kaggle/input/datasets/aayushjamwal03/prompts/prompts_hindi.json", "r", encoding="utf-8") as f:
    hindi_prompts = json.load(f)

all_items = []
for idx, item in enumerate(english_prompts[:30], 1):
    if isinstance(item, str):
        all_items.append({"id": f"EN_{idx:02d}", "language": "en", "prompt": item, "acceptable_answers": []})
    else:
        all_items.append({
            "id": item.get("id", f"EN_{idx:02d}"),
            "language": "en",
            "prompt": item["prompt"],
            "acceptable_answers": item.get("acceptable_answers", []),
        })

for idx, item in enumerate(hindi_prompts[:30], 1):
    if isinstance(item, str):
        all_items.append({"id": f"HI_{idx:02d}", "language": "hi", "prompt": item, "acceptable_answers": []})
    else:
        all_items.append({
            "id": item.get("id", f"HI_{idx:02d}"),
            "language": "hi",
            "prompt": item["prompt"],
            "acceptable_answers": item.get("acceptable_answers", []),
        })

print(f"Loaded {len(all_items)} total prompts (30 EN + 30 HI).")

Loaded 60 total prompts (30 EN + 30 HI).


# Prompts through GEMINI API

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(
    api_key="GEMINI_KEY_HERE",
    http_options={"api_version": "v1beta"},
)

gemini_results = []
gemini_total_cost = 0.0

for i in all_prompts:
    start = time.time()
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite" ,
        contents=i["prompt"],
        config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT),
    )
    latency_ms = (time.time() - start) * 1000
    output = response.text

    # Check if expected keyword appears in the model's output
    expected = i.get("expected_answer") or i.get("expected_disease") or ""
    is_correct = expected.lower() in output.lower()

    # Gemini 2.5 Flash pricing: $0.30 per 1M input tokens, $2.50 per 1M output tokens
    in_tokens = response.usage_metadata.prompt_token_count
    out_tokens = response.usage_metadata.candidates_token_count
    gemini_total_cost += (in_tokens * 0.30 / 1_000_000) + (out_tokens * 2.50 / 1_000_000)

    gemini_results.append({
        "id": i["id"],
        "correct": is_correct,
        "latency_ms": round(latency_ms, 1),
        "output": output
    })
    print(f"Gemini | {i['id']} | {latency_ms:.0f} ms | Correct: {is_correct}")
    time.sleep(6.5)  # Stay under 10 requests/minute on free tier

In [ ]:
import json
from IPython.display import FileLink

# 1. Grab the Gemini results from whichever variable or progress file exists
if "gemini_results" in globals():
    data = gemini_results
elif "gemini_logs" in globals():
    data = gemini_logs
else:
    with open("/kaggle/working/gemini_progress.json", "r", encoding="utf-8") as f:
        data = json.load(f)

# 2. Save cleanly to Kaggle's writable working directory
output_path = "/kaggle/working/gemini_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(data)} Gemini results to {output_path}!")

# 3. Display a direct download link in Kaggle
FileLink("gemini_results.json")

# LOADING SARVAM 30B and RUNNING

In [ ]:
import os, glob, ctypes, re, json, time
from IPython.display import display, FileLink
from huggingface_hub import HfApi, hf_hub_download

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["TMPDIR"] = "/tmp"

# 1. Preload CUDA 12 shared libraries before importing llama_cpp
def preload_cuda_lib(lib_pattern, pip_pkg):
    matches = sorted(
        glob.glob(f"/usr/local/**/{lib_pattern}", recursive=True)
        + glob.glob(f"/usr/lib/**/{lib_pattern}", recursive=True)
    )
    if not matches:
        os.system(f"pip install -q --no-deps {pip_pkg}")
        matches = sorted(glob.glob(f"/usr/local/**/{lib_pattern}", recursive=True))
    matches.sort(key=len)
    print(f"Preloading {lib_pattern} -> {matches[0]}", flush=True)
    ctypes.CDLL(matches[0], mode=ctypes.RTLD_GLOBAL)

preload_cuda_lib("libcudart.so.12*", "nvidia-cuda-runtime-cu12")
preload_cuda_lib("libcublasLt.so.12*", "nvidia-cublas-cu12")
preload_cuda_lib("libcublas.so.12*", "nvidia-cublas-cu12")

from llama_cpp import Llama

# 2. Ensure all 6 shards of sarvam-30b-Q4_K_M.gguf are in /tmp/sarvam_gguf
repo_id = "sarvamai/sarvam-30b-gguf"
api = HfApi()
files = api.list_repo_files(repo_id)
parts = sorted([f for f in files if "sarvam-30b-Q4_K_M.gguf-" in f and f.endswith(".gguf")])

for idx, part_file in enumerate(parts, 1):
    local_file = os.path.join("/tmp/sarvam_gguf", os.path.basename(part_file))
    if not os.path.exists(local_file):
        print(f"  [{idx}/6] Downloading {part_file}...", flush=True)
        hf_hub_download(repo_id=repo_id, filename=part_file, local_dir="/tmp/sarvam_gguf")
    else:
        print(f"  [{idx}/6] Already present: {part_file}", flush=True)

model_path = os.path.join("/tmp/sarvam_gguf", os.path.basename(parts[0]))

# 3. Load Sarvam-30B across both T4 GPUs
llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,          # Offload 100% of layers across both T4 GPUs
    tensor_split=[0.5, 0.5],  # Split evenly across GPU 0 and GPU 1
    n_ctx=2048,
    verbose=False,
)
print("Sarvam-30B GGUF loaded onto both T4 GPUs! Starting 60-prompt inference...", flush=True)

# 4. Run 60 prompts, strip <think> blocks, and save exact Qwen/Gemini 4-key schema
SYSTEM_PROMPT = (
    "You are an expert Indian agricultural advisor for AgriVision. "
    "Be concise and direct. Diagnose crop diseases accurately, suggest practical treatment, "
    "and respond in the exact language (English or Hindi) of the farmer's query."
)

def strip_thinking(text):
    cleaned = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    if cleaned.startswith("<think>"):
        return ""
    return cleaned

sarvam_results = []
for idx, item in enumerate(all_items, 1):
    t0 = time.perf_counter()
    resp = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item["prompt"]},
        ],
        max_tokens=1024,      # Gives Sarvam-30B enough tokens to finish <think> and write the final answer
        temperature=0.0,
    )
    latency_ms = round((time.perf_counter() - t0) * 1000, 1)

    raw_reply = resp["choices"][0]["message"]["content"].strip()
    final_reply = strip_thinking(raw_reply)

    # Exact 4-key schema matching qwen_results.json and gemini_results.json
    sarvam_results.append({
        "id": item["id"],
        "correct": False,     # Will be graded alongside Qwen & Gemini in Cell 4
        "latency_ms": latency_ms,
        "output": final_reply,
    })

    if idx % 5 == 0 or idx == len(all_items):
        print(f"[{idx}/60] Done | Last latency: {latency_ms:.1f} ms | Clean output chars: {len(final_reply)}", flush=True)

out_path = "/kaggle/working/sarvam_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(sarvam_results, f, ensure_ascii=False, indent=2)

avg_ms = sum(r["latency_ms"] for r in sarvam_results) / len(sarvam_results)
print(f"\nSaved 60 clean responses to {out_path} | Avg Latency: {avg_ms:.1f} ms")
display(FileLink("sarvam_results.json"))

Preloading libcudart.so.12* -> /usr/local/lib/python3.13/dist-packages/nvidia/cuda_runtime/lib/libcudart.so.12
Preloading libcublasLt.so.12* -> /usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib/libcublasLt.so.12
Preloading libcublas.so.12* -> /usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib/libcublas.so.12
  [1/6] Already present: sarvam-30b-Q4_K_M.gguf-00001-of-00006.gguf
  [2/6] Already present: sarvam-30b-Q4_K_M.gguf-00002-of-00006.gguf
  [3/6] Already present: sarvam-30b-Q4_K_M.gguf-00003-of-00006.gguf
  [4/6] Already present: sarvam-30b-Q4_K_M.gguf-00004-of-00006.gguf
  [5/6] Already present: sarvam-30b-Q4_K_M.gguf-00005-of-00006.gguf
  [6/6] Already present: sarvam-30b-Q4_K_M.gguf-00006-of-00006.gguf
Sarvam-30B GGUF loaded onto both T4 GPUs! Starting 60-prompt inference...
[5/60] Done | Last latency: 16711.0 ms | Clean output chars: 0
[10/60] Done | Last latency: 16706.8 ms | Clean output chars: 0
[15/60] Done | Last latency: 16600.1 ms | Clean output chars: 0
[2

/kaggle/working/sarvam_results.json

In [ ]:
import re, json, time
from jinja2 import Environment
from IPython.display import display, FileLink

# 1. Inspect the exact assistant prefix produced by Sarvam-30B's native chat template
raw_template = llm.metadata.get("tokenizer.chat_template", "")
bos_token = llm.metadata.get("tokenizer.ggml.bos_token", "")
eos_token = llm.metadata.get("tokenizer.ggml.eos_token", "")

jinja_env = Environment()
template = jinja_env.from_string(raw_template)

def format_sarvam_force_no_think(system_msg, user_msg):
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]
    rendered = template.render(
        messages=messages,
        add_generation_prompt=True,
        bos_token=bos_token,
        eos_token=eos_token,
    )
    # Force-close the thinking block immediately so Sarvam-30B skips reasoning and outputs the answer
    if "<think>" in rendered and "</think>" not in rendered.split("<think>")[-1]:
        rendered = rendered + "\n</think>\n\n"
    elif "</think>" not in rendered[-30:]:
        rendered = rendered + "<think>\n</think>\n\n"
    return rendered

SYSTEM_PROMPT = (
    "You are an expert Indian agricultural advisor for AgriVision. "
    "Diagnose crop diseases accurately, suggest practical treatment, "
    "and respond concisely in the exact language (English or Hindi) of the farmer's query."
)

# 2. 3-second test on EN_01 to confirm zero <think> tokens
sample_prompt = format_sarvam_force_no_think(SYSTEM_PROMPT, all_items[0]["prompt"])
print("Prompt suffix sent to model:", repr(sample_prompt[-80:]), flush=True)

test_out = llm(sample_prompt, max_tokens=250, temperature=0.0, stop=[eos_token] if eos_token else None)
print("EN_01 direct output:\n", test_out["choices"][0]["text"].strip(), "\n" + "-"*50, flush=True)

# 3. Run all 60 prompts and save exact Qwen/Gemini 4-key schema
sarvam_results = []
for idx, item in enumerate(all_items, 1):
    prompt_str = format_sarvam_force_no_think(SYSTEM_PROMPT, item["prompt"])

    t0 = time.perf_counter()
    resp = llm(
        prompt_str,
        max_tokens=300,
        temperature=0.0,
        stop=[eos_token] if eos_token else None,
    )
    latency_ms = round((time.perf_counter() - t0) * 1000, 1)

    reply = resp["choices"][0]["text"].strip()
    # Strip any residual thinking tags just in case
    reply = re.sub(r"<think>.*?</think>", "", reply, flags=re.DOTALL).strip()
    reply = reply.replace("</think>", "").strip()

    sarvam_results.append({
        "id": item["id"],
        "correct": False,  # Graded in Cell 4
        "latency_ms": latency_ms,
        "output": reply,
    })

    if idx % 5 == 0 or idx == len(all_items):
        print(f"[{idx}/60] Done | Last latency: {latency_ms:.1f} ms | Output chars: {len(reply)}", flush=True)

out_path = "/kaggle/working/sarvam_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(sarvam_results, f, ensure_ascii=False, indent=2)

avg_ms = sum(r["latency_ms"] for r in sarvam_results) / len(sarvam_results)
print(f"\nSaved 60 clean responses to {out_path} | Avg Latency: {avg_ms:.1f} ms")
display(FileLink("sarvam_results.json"))

Prompt suffix sent to model: 'nd a treatment.<|end_of_turn|>\n<|start_of_turn|><|assistant|>\n<think>\n</think>\n\n'
EN_01 direct output:
 **Disease Identification:**
The symptoms you're describing—dark concentric rings and yellowing around the spots on tomato leaves—are characteristic of **Early Blight**.

**Treatment Recommendation:**

1.  **Remove Infected Parts:** Immediately remove and destroy any leaves with dark spots. This stops the spread.
2.  **Improve Air Circulation:** Prune lower leaves to improve airflow, which helps prevent fungal growth.
3.  **Apply Fungicide:** Spray a suitable fungicide. A good option is **Mancozeb** or **Copper Oxychloride**.
    *   **Mancozeb:** Apply at a rate of 2.5-3.0 grams per liter of water.
    *   **Copper Oxychloride:** Apply at a rate of 2.5-3.0 grams per liter of water.
4.  **Water Management:** Water the soil, not the leaves, to keep the foliage dry.

**Important:** Always follow the manufacturer's instructions on the product label for 

/kaggle/working/sarvam_results.json

In [ ]:
import json, re
import pandas as pd
from IPython.display import display, FileLink

EVAL_SUITE_PATH = "/kaggle/input/datasets/aayushjamwal03/eval-updated/eval_suite.json"
MODEL_PATHS = {
    "Qwen3-8B":   "/kaggle/input/datasets/aayushjamwal03/results-eval-suite/qwen_results.json",
    "Gemini":     "/kaggle/input/datasets/aayushjamwal03/results-eval-suite/gemini_results.json",
    "Sarvam-30B": "/kaggle/working/sarvam_results.json",
}

def load_eval_suite_safe(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read().strip()
    try:
        return json.loads(raw, strict=False)
    except json.JSONDecodeError:
        # Recover all complete JSON objects if the file was truncated near the end
        last_complete = raw.rfind("},")
        if last_complete != -1:
            repaired = raw[:last_complete + 1] + "\n]"
            items = json.loads(repaired, strict=False)
            print(f"Warning: eval_suite.json was truncated on disk; recovered {len(items)} complete ground-truth items.")
            return items
        raise

suite_raw = load_eval_suite_safe(EVAL_SUITE_PATH)
suite_by_id = {item["id"]: item for item in suite_raw}

# 1. Score sarvam_results.json against eval_suite.json and save the final 4-key file
with open(MODEL_PATHS["Sarvam-30B"], "r", encoding="utf-8") as f:
    sarvam_rows = json.load(f)

for r in sarvam_rows:
    gt = suite_by_id.get(r["id"], {})
    acceptable = gt.get("acceptable_answers", [])
    reply_low = r["output"].lower()
    r["correct"] = any(term.lower() in reply_low for term in acceptable) if acceptable else False

with open("/kaggle/working/sarvam_results_evaluated.json", "w", encoding="utf-8") as f:
    json.dump(sarvam_rows, f, ensure_ascii=False, indent=2)

MODEL_PATHS["Sarvam-30B"] = "/kaggle/working/sarvam_results_evaluated.json"

# 2. Compare all 3 models side-by-side
summary_rows = []
aligned_by_id = {}

for model_name, path in MODEL_PATHS.items():
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    # Evaluate only on IDs present in eval_suite.json so all 3 models are compared on the exact same subset
    scored = [r for r in records if r["id"] in suite_by_id]
    en_rows = [r for r in scored if r["id"].startswith("EN_")]
    hi_rows = [r for r in scored if r["id"].startswith("HI_")]
    latencies = [float(r["latency_ms"]) for r in records]

    summary_rows.append({
        "Model": model_name,
        "Graded Prompts": len(scored),
        "Overall Acc (%)": round(sum(int(r["correct"]) for r in scored) / len(scored) * 100, 2),
        "EN Acc (%)": round(sum(int(r["correct"]) for r in en_rows) / len(en_rows) * 100, 2) if en_rows else "N/A",
        "HI Acc (%)": round(sum(int(r["correct"]) for r in hi_rows) / len(hi_rows) * 100, 2) if hi_rows else "N/A",
        "Avg Latency (ms)": round(sum(latencies) / len(latencies), 1),
        "Median Latency (ms)": round(pd.Series(latencies).median(), 1),
    })

    for r in records:
        pid = r["id"]
        gt = suite_by_id.get(pid, {})
        aligned_by_id.setdefault(pid, {
            "id": pid,
            "language": "en" if pid.startswith("EN_") else "hi",
            "prompt": gt.get("prompt", ""),
        })
        aligned_by_id[pid][f"{model_name}_correct"] = r["correct"]
        aligned_by_id[pid][f"{model_name}_latency_ms"] = r["latency_ms"]
        aligned_by_id[pid][f"{model_name}_output"] = r["output"]

summary_df = pd.DataFrame(summary_rows)
print("=== AGRIVISION 3-MODEL BENCHMARK REPORT ===")
display(summary_df)

comp_df = pd.DataFrame(list(aligned_by_id.values()))
comp_df.to_csv("/kaggle/working/eval_comparison_all3.csv", index=False, encoding="utf-8-sig")

print("Saved:")
display(FileLink("sarvam_results_evaluated.json"))
display(FileLink("eval_comparison_all3.csv"))

=== AGRIVISION 3-MODEL BENCHMARK REPORT ===


,Model,Graded Prompts,Overall Acc (%),EN Acc (%),HI Acc (%),Avg Latency (ms),Median Latency (ms)
0,Qwen3-8B,60,38.33,53.33,23.33,13765.0,16345.2
1,Gemini,60,81.67,76.67,86.67,2900.6,1252.8
2,Sarvam-30B,60,55.00,63.33,46.67,4483.7,4478.0


Saved:


/kaggle/working/sarvam_results_evaluated.json

/kaggle/working/eval_comparison_all3.csv

In [ ]:
import json
import pandas as pd

# 1. Load your comparison CSV and your 60-item eval_suite.json
df = pd.read_csv("/kaggle/working/eval_comparison_all3.csv")

# Update path below to wherever your 60-item eval_suite.json is stored
with open("/kaggle/input/datasets/aayushjamwal03/eval-updated/eval_suite.json", "r", encoding="utf-8") as f:
    suite = {item["id"]: item["acceptable_answers"] for item in json.load(f)}

pass_col = "Sarvam-30B_pass" if "Sarvam-30B_pass" in df.columns else "Sarvam-30B_correct"

# 2. Check Hindi items marked FALSE (to spot False Negatives vs. real mistakes)
hi_misses = df[(df["language"] == "hi") & (df[pass_col] == False)]
print(f"=== SARVAM HINDI MARKED FALSE ({len(hi_misses)} / 30) ===\n")

for _, row in hi_misses.iterrows():
    pid = row["id"]
    print(f"[{pid}] Prompt: {row['prompt']}")
    print(f"   Expected Keywords : {suite.get(pid, [])}")
    print(f"   Sarvam Output     : {str(row['Sarvam-30B_output'])[:280].replace(chr(10), ' ')}")
    print("-" * 70)

# 3. Also check items marked TRUE with very short outputs (to spot False Positives)
hi_passes = df[(df["language"] == "hi") & (df[pass_col] == True)]
print(f"\n=== SARVAM HINDI MARKED TRUE ({len(hi_passes)} / 30) — QUICK SANITY CHECK ===\n")
for _, row in hi_passes.iterrows():
    pid = row["id"]
    print(f"[{pid}] Expected: {suite.get(pid, [])[:3]} | Output: {str(row['Sarvam-30B_output'])[:160].replace(chr(10), ' ')}")

=== SARVAM HINDI MARKED FALSE (16 / 30) ===

[HI_01] Prompt: गेहूं की पत्तियों पर पीले रंग की धारियां दिख रही हैं। यह कौन सा रोग है और इसका क्या उपाय है?
   Expected Keywords : ['yellow rust', 'stripe rust', 'puccinia striiformis', 'linear rust', 'पीला रतुआ', 'धारीदार रतुआ', 'पीली गेरुई', 'पीला किट्ट', 'रतुआ']
   Sarvam Output     : गेहूं की पत्तियों पर पीले रंग की धारियां दिखना, **गेहूं का पीला मोज़ेक रोग (Wheat Yellow Mosaic Disease)** है। यह एक वायरल रोग है जो सफेद मक्खी (Whitefly) के माध्यम से फैलता है।  **उपाय:**  1.  **रोग की पहचान:** यह रोग पत्तियों पर पीले, सफेद या हरे रंग की धारियां या धब्बे बनाकर द
----------------------------------------------------------------------
[HI_02] Prompt: टमाटर की पत्तियों पर गहरे भूरे रंग के गोल छल्ले और चारों ओर पीलापन दिख रहा है। रोग की पहचान करें और उपचार बताएं।
   Expected Keywords : ['early blight', 'alternaria solani', 'alternaria leaf spot', 'target spot', 'अगेती झुलसा', 'अगेती अंगमारी']
   Sarvam Output     : **रोग की पहचान:** यह लक्षण **